#Libraries

In [17]:
!pip install pyngrok


In [18]:
!pip install -q --upgrade bitsandbytes


In [19]:
!pip install -q --upgrade transformers


In [20]:
!pip install gradio PyPDF2 sentence-transformers faiss-cpu transformers langchain pyngrok

In [21]:
!pip install gradio PyPDF2 sentence-transformers faiss-cpu transformers langchain langchain-community pyngrok

In [22]:
!pip install -q gradio PyPDF2 transformers accelerate sentence-transformers faiss-cpu bitsandbytes

In [23]:
!pip install gradio PyPDF2 sentence-transformers faiss-cpu transformers langchain langchain-community pyngrok -q

import gradio as gr
import PyPDF2
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.llms import HuggingFacePipeline
from transformers import pipeline
from pyngrok import ngrok
import nest_asyncio
import socket
from fastapi import FastAPI, UploadFile, Form
import uvicorn
import threading
import shutil
import os

# Embeddings

In [24]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [25]:
db = None
llm = None


# LLM

In [26]:
qa_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=512,
    temperature=0.8,  # creative answers
    top_p=0.9,
)
llm = HuggingFacePipeline(pipeline=qa_pipeline)


Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


# Load PDF

In [27]:
def load_pdf(pdf_file):
    global db
    reader = PyPDF2.PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text.strip() + " "

    if not text:
        return " No text could be extracted from this PDF. Make sure it is not scanned."

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_text(text)

    db = FAISS.from_texts(chunks, embedding_model)
    return " PDF uploaded & processed successfully!"

# Answer Questions

In [28]:
from fastapi import FastAPI, UploadFile, File

def answer_question(question):
    global db
    if db is None:
        return "⚠️ Please upload a PDF first."

    # Retrieve top 3 relevant chunks
    docs = db.similarity_search(question, k=3)
    context = "\n".join([doc.page_content for doc in docs])

    # Adaptive factual/creative check
    # If the question words appear in context → factual
    is_factual = any(word.lower() in context.lower() for word in question.split())

    if is_factual:
        prompt = f"""
You are a helpful assistant. Answer ONLY using the context below.
Do NOT add any information that is not present in the context.
If the answer is not available, say: "The answer is not available in the document."

Context:
{context}

Question: {question}
Answer:
"""
    else:
        prompt = f"""
You are a helpful assistant. Answer using the context below.
You can be creative and blend relevant insights or related info.
Do not invent unrelated facts.

Context:
{context}

Question: {question}
Answer:
"""

    result = llm(prompt)
    if isinstance(result, list):
        return result[0]['generated_text']
    elif isinstance(result, str):
        return result
    else:
        return str(result)

# =========================
# 🚀 FastAPI Setup ⭐ NEW
# =========================
app = FastAPI()

# Root route (so opening ngrok link shows something instead of blank page)
@app.get("/")   # ⭐ NEW
async def root():
    return {"message": "✅ FastAPI PDF Assistant is running! Use /upload_pdf and /ask_question endpoints."}

# Health check route
@app.get("/status")   # ⭐ NEW
async def status():
    return {"status": "connected", "db_ready": db is not None}

# Upload PDF
@app.post("/upload_pdf")
@app.post("/upload-pdf/")
async def upload_pdf_file(file: UploadFile = File(...)):
    # Save the uploaded file to a directory
    upload_dir = "uploaded_pdfs"
    os.makedirs(upload_dir, exist_ok=True)

    file_path = os.path.join(upload_dir, file.filename)

    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    return {"filename": file.filename, "status": "uploaded successfully"}

# Ask a question
@app.post("/ask_question")
async def ask_question_api(question: str = Form(...)):
    answer = answer_question(question)
    return {"question": question, "answer": answer}


# Gradio UI

In [29]:
with gr.Blocks(css="""
    /* Full-screen background for Gradio container */
    .gradio-container {
        height: 100vh;
        width: 100vw;
        background-image: url('https://images.unsplash.com/photo-1504384308090-c894fdcc538d?auto=format&fit=crop&w=1600&q=80');
        background-size: cover;
        background-position: center;
        background-attachment: fixed;
        display: flex;
        justify-content: center;
        align-items: flex-start;
        overflow-y: auto;
        padding: 50px 0;
        font-family: Arial, sans-serif;
    }

    /* Title */
    #title {
        text-align: center;
        font-size: 42px;
        font-weight: bold;
        margin-bottom: 40px;
        color: #fff;
        text-shadow: 3px 3px 12px rgba(0,0,0,0.7);
        animation: fadeInDown 1s ease-out;
    }

    @keyframes fadeInDown {
        0% { opacity: 0; transform: translateY(-50px);}
        100% { opacity: 1; transform: translateY(0);}
    }

    /* Sections */
    .pdf-section, .qa-section {
        background-color: rgba(0, 0, 0, 0.7);
        padding: 30px;
        border-radius: 20px;
        margin: 20px auto;
        width: 90%;
        max-width: 800px;
        box-shadow: 0 8px 20px rgba(0,0,0,0.6);
        animation: fadeInUp 1.2s ease-out;
    }

    @keyframes fadeInUp {
        0% { opacity: 0; transform: translateY(50px);}
        100% { opacity: 1; transform: translateY(0);}
    }

    /* Buttons */
    .gr-button {
        background-color: #4a90e2;
        color: white;
        font-weight: bold;
        border-radius: 10px;
        transition: all 0.3s ease;
    }

    .gr-button:hover {
        background-color: #357ABD;
        transform: scale(1.05);
    }

    /* Textboxes */
    .gr-textbox {
        background-color: rgba(255, 255, 255, 0.95);
        color: black;
        border-radius: 10px;
        padding: 10px;
        font-size: 16px;
        transition: box-shadow 0.3s ease;
    }

    .gr-textbox:focus {
        box-shadow: 0 0 15px rgba(74, 144, 226, 0.7);
    }

""") as demo:

    gr.Markdown("##  AI PDF Assistant – Creative & Adaptive Chat", elem_id="title")

    # PDF Upload Section
    with gr.Group(elem_classes="pdf-section"):
        gr.Markdown("### Step 1: Upload your PDF")
        pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
        upload_btn = gr.Button("Process PDF")
        status = gr.Textbox(label="Status", interactive=False)

    # Question & Answer Section
    with gr.Group(elem_classes="qa-section"):
        gr.Markdown("### Step 2: Ask questions about your PDF")
        question = gr.Textbox(label="Type your question here")
        ask_btn = gr.Button("Get Answer")
        answer = gr.Textbox(label="Answer", interactive=False)

    # Connect functionality
    upload_btn.click(load_pdf, inputs=pdf_input, outputs=status)
    ask_btn.click(answer_question, inputs=question, outputs=answer)


In [30]:
from pyngrok import ngrok
ngrok.kill()  # kills all old tunnels


# Launch with ngrok

In [31]:
NGROK_AUTH_TOKEN = "2vAU7BnUA6Fn0NrbCqODXwu6tKf_6Uxvnd7hizCKBQauVxDPM"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()  # Kill old tunnels
def find_free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

# 🔹 Allocate free ports
fastapi_port = find_free_port()
gradio_port = find_free_port()

# 🔹 Ngrok tunnels for these ports
fastapi_url = ngrok.connect(fastapi_port, "http")
gradio_url  = ngrok.connect(gradio_port, "http")

print("🌍 FastAPI Public URL:", fastapi_url.public_url)
print("🌍 Gradio  Public URL:", gradio_url.public_url)

# 🔹 Fix asyncio loop conflicts in Colab
nest_asyncio.apply()

# 🔹 Run FastAPI in background
def run_fastapi():
    uvicorn.run(app, host="0.0.0.0", port=fastapi_port)

threading.Thread(target=run_fastapi, daemon=True).start()

# 🔹 Run Gradio
demo.launch(server_name="0.0.0.0", server_port=gradio_port, share=False)

🌍 FastAPI Public URL: https://7176b9256286.ngrok-free.app
🌍 Gradio  Public URL: https://910d90cfb753.ngrok-free.app


INFO:     Started server process [664]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:51561 (Press CTRL+C to quit)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>